# Coding attention mechanisms


### A simple self-attention mechanism without trainable weights

In [218]:
import torch
## this just represent a list of token embeddins.
inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your     (x^1)
    [0.55, 0.87, 0.66], # journey  (x^2)
    [0.57, 0.85, 0.64], # starts   (x^3)
    [0.22, 0.58, 0.33], # with     (x^4)
    [0.77, 0.25, 0.10], # one      (x^5)
    [0.05, 0.80, 0.55] # step     (x^6)
])

In [180]:
query = inputs[1] # this is the token embeddings for the word --journey--
attn_scores_2 = torch.empty(inputs.shape[0]) # must be equal to 6
print("attention scores 2", attn_scores_2)
for i, x_i in enumerate(inputs):    
    weight = torch.dot(query, x_i)
    attn_scores_2[i] = weight

print(attn_scores_2)

attention scores 2 tensor([-1.1242e+34,  1.9800e-42,  7.0485e-43,  0.0000e+00,  1.2472e-42,
         0.0000e+00])
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [181]:
attn_weights_2_temp_normalization = attn_scores_2 / attn_scores_2.sum()
print("Attentions weights: ", attn_weights_2_temp_normalization)
print("Sum", attn_weights_2_temp_normalization.sum())

Attentions weights:  tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum tensor(1.0000)


In [182]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_wieghts_2_normalization_naive = softmax_naive(attn_scores_2)
print("Attention weights: ", attn_wieghts_2_normalization_naive)
print("Sum", attn_wieghts_2_normalization_naive.sum())

Attention weights:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum tensor(1.)


In [183]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


1. first we calculate the weight of each token related to each token in the sequence.
2. then we normalize those weight using the softmax function so that the sum up 1.
3. we calculate the context vector context_vector(n) in this case context_vector(1) by
    3.1 multipliying the weight_12 related to the current token in the sequence to its corresponding token embedding
    3.2 the result of multiplying the embedding token by it is weight related to one token in the sequence it is added to the next 
    token embedding * wieght and then we have the sum of all the weighted embeddins and that is the context vector for the token (i) 


In [184]:
query = inputs[1]
context_vector_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vector_2 += attn_weights_2[i] * x_i

print("Context vector for the word: journey", context_vector_2)

Context vector for the word: journey tensor([0.4419, 0.6515, 0.5683])


In [185]:
# the result of the attention scores it is a matrix with a shape of 6*6 elements
attn_scores = torch.zeros(inputs.shape[0], inputs.shape[0])
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [186]:
inputs_transpose = inputs.T
print(inputs)
print(inputs_transpose)

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])
tensor([[0.4300, 0.5500, 0.5700, 0.2200, 0.7700, 0.0500],
        [0.1500, 0.8700, 0.8500, 0.5800, 0.2500, 0.8000],
        [0.8900, 0.6600, 0.6400, 0.3300, 0.1000, 0.5500]])


In [187]:
# 1. compute the attention scores as dot products between the inputs.
# 2. the attentions weights are the normalized version of the attention scores
# 3. the context vectors are computed as weighted sum over the inputs. weight_1 * embedding_1 + weight_2 * embedding_2 ... weight_n + embedding_n
attn_scores = inputs @ inputs_transpose
attn_scores = torch.softmax(attn_scores, dim=-1)
print(attn_scores[1])
print(sum(attn_scores[0]))
print(attn_scores.sum(dim=-1))

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.0000)
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [188]:
# in thee third and final step, we use these attentation wieghts to compute all contexxt vectors via matrix multiplication.
all_context_vecs = attn_scores @ inputs

In [189]:
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [190]:
print("Context vector for the word: journey", context_vector_2)

Context vector for the word: journey tensor([0.4419, 0.6515, 0.5683])


# Implementing self-attention with trainable weights
our next step will be to implement the self-attention mechanism used in the original transformer architecture, the GPT
models, and most other popular LLMs. this self-attention mechanism is also called
**_scaled dot-product attention_** 

we will now extend the self-attention mechanism with trainable weights.

In [191]:
# we introduce 3 weights
# wq (weight for query)
# wk (weight for key)
# wv (weight for value)

x_2 = inputs[1] # represents: journey
d_in = inputs.shape[1]
d_out = 2

In [192]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [193]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)


tensor([0.4306, 1.4551])


In [194]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape", keys.shape)
print("values.shape", values.shape)

keys.shape torch.Size([6, 2])
values.shape torch.Size([6, 2])


In [195]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


In [196]:
attn_score_22 = query_2 @ keys.T
print("shape of query_2", query_2.shape)
print("shape of the keys", keys.shape)

shape of query_2 torch.Size([2])
shape of the keys torch.Size([6, 2])


In [197]:
print(attn_score_22)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [198]:
embedding_dimesion_of_the_keys = keys.shape[-1]
print(embedding_dimesion_of_the_keys)

2


In [199]:
attn_Weights_2 = torch.softmax(attn_score_22 / embedding_dimesion_of_the_keys**0.5, dim=-1)
print(attn_score_22)
print(attn_weights_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])
tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


In [200]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)
print(attn_score_22)
d_k = keys.shape[-1]
print(d_k)
print(embedding_dimesion_of_the_keys)
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
attn_Weights_22 = torch.softmax(attn_score_22 / embedding_dimesion_of_the_keys**0.5, dim=-1)
print(attn_weights_2)
print(attn_Weights_22)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])
tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])
2
2
tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [201]:
context_vector_2 = attn_Weights_22 @ values
print(attn_Weights_22.shape)
print(values.shape)
print(context_vector_2)

torch.Size([6])
torch.Size([6, 2])
tensor([0.3061, 0.8210])


# Calculating the context vector by using query, key and value
- we need to set the trainable weights for the query, key and value based on the dim_in and dim_out, usually the dim of the embeddings.
- we need to calculate the the attentions scores for each token in the input sequence
    - we use the current token, *token_n* as query for calculating the attention score related to this token and the rest of the sequence
    - we multiple the current query against the key for each token and we got the scores
    - we calculate the weights by applying the softmax function, but we divided the scores between the the square root of the dimesion of
    the embeddings. 
    - finally we multiple those weights for each value of each token of in the sequence, and that returns the context_vector for the
    given token, token_n in the sequence.

# Calculating a context vector with scaled dot-product attention

- Start with input token embeddings. Each token in the input sequence is represented by an embedding vector.

- Create three trainable weight matrices:

  - `W_query`
  - `W_key`
  - `W_value`

- These matrices project each input token embedding into three different vectors:

  - a query vector
  - a key vector
  - a value vector

  For example:

  ```python
  queries = inputs @ W_query
  keys    = inputs @ W_key
  values  = inputs @ W_value
  ```

- To compute the context vector for a specific token, use that token’s query vector.

- The query represents the current token we are focusing on.

- Compare this query vector with the key vector of every token in the sequence.

- This comparison is done using dot products:

  ```python
  attn_scores = query @ keys.T
  ```

- The result is a set of attention scores.

- These attention scores measure how strongly the current token should attend to each token in the sequence.

- A higher dot product means the query and key are more similar, so the corresponding token receives a higher attention score.

- Convert the attention scores into attention weights using the softmax function.

- Before applying softmax, divide the attention scores by the square root of the key dimension, usually written as `d_k`:

  ```python
  attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)
  ```

- This scaling step prevents the dot products from becoming too large.

- If the dot products become too large, the softmax function can produce very sharp values, which can lead to very small gradients and make training slower or unstable.

- The scaling by the square root of the key dimension is why this mechanism is called **scaled dot-product attention**.

- The attention weights tell us how much importance to assign to each token’s value vector.

- Finally, compute the context vector as a weighted sum of the value vectors:

  ```python
  context_vec = attn_weights @ values
  ```

- The result is the context vector for the current token.

- This context vector contains information from the current token and the other tokens in the sequence, weighted by their relevance to the current token.

- In compact mathematical form, scaled dot-product attention is:

  ```text
  softmax(QKᵀ / sqrt(d_k))V
  ```

  where:

  - `Q` is the query matrix
  - `K` is the key matrix
  - `V` is the value matrix
  - `d_k` is the dimension of the key vectors



In [202]:
import torch.nn as nn
import torch
class SelfAttention_v1(nn.Module):
    def __init__(self, sa_v2):
        super().__init__()
        self.W_query  = nn.Parameter(sa_v2.W_query.weight.T)
        self.W_key    = nn.Parameter(sa_v2.W_key.weight.T)
        self.W_value  = nn.Parameter(sa_v2.W_value.weight.T)
    
    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vector = attn_weights @ values
        return context_vector

In [203]:
# print(d_in, d_out)
# torch.manual_seed(123)
# sa_v1 = SelfAttention_v1(d_in, d_out)
# print(sa_v1(inputs))

In [204]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key    = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value  = nn.Linear(d_in, d_out, bias=qkv_bias)
        # self.initial_weights = self.W_key.
    
    def forward(self, x):
        keys = self.W_key(x)
        values = self.W_value(x)
        queries =self.W_query(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vector = attn_weights @ values
        return context_vector

In [205]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

sa_v1 = SelfAttention_v1(sa_v2=sa_v2)
print(sa_v1(inputs))
# print(torch.rand(d_in, d_out))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


# Hiding future words with causal attention
Hiding future words with causal attention mechanism


In [206]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights_2 = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights_2)


tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [207]:
context_lenght = attn_scores.shape[-1]
mask_simple = torch.tril(torch.ones(context_lenght, context_lenght))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [208]:
masked_simple = attn_weights_2 * mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [209]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [210]:
mask = torch.triu(torch.ones(context_lenght, context_lenght), diagonal=1) # diagonal=1 means keep values below the main diagonal.
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked) 

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [211]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [212]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6,6)
print(dropout(example))



tensor([[2., 2., 2., 2., 2., 2.],
        [0., 2., 0., 0., 0., 0.],
        [0., 0., 2., 0., 2., 0.],
        [2., 2., 0., 0., 0., 2.],
        [2., 0., 0., 0., 0., 2.],
        [0., 2., 0., 0., 0., 0.]])


In [213]:
torch.manual_seed(123)
print(dropout(attn_weights))


tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.8966, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4921, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4350, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.0000, 0.0000, 0.0000, 0.0000]],
       grad_fn=<MulBackward0>)


In [214]:
class CauslaAttention(nn.Module):
    def __init__(self, d_in, d_out, context_lenght, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_lenght, context_lenght), diagonal=1) 
        )

    def forward(self, x):
        # explodes the tensor shape 
        # b is the number of batches in this step.
        # num_tokens are the number of words in per batch. 
        # d_in it is the number of dimesions of the embeddings that represent each token. 
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vector = attn_weights @ values
        return context_vector

    


In [215]:
!uv add importnb

Resolved 92 packages in 2ms
Audited 77 packages in 0.24ms


In [216]:
# import importnb
# with importnb.Notebook(include_non_defs=False):
#     from chapter2 import create_dataloader_v1

# with open("training_text.txt", "r", encoding="utf-8") as file:
#     txt = file.read()

# max_length = 4
# dataloader = create_dataloader_v1(txt, batch_size=8, max_length=max_length, stride=max_length, suffle=False)
# data_iter = iter(dataloader)
# batch = next(data_iter)
# inputs, targets = batch
# print(inputs)

In [220]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch)

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])


In [225]:
torch.manual_seed(123)
context_lenght = batch.shape[1]
ca = CauslaAttention(d_in, d_out, context_lenght, 0.0)
context_vector = ca(batch)
print(context_vector)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)


In [226]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_lenght, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                CauslaAttention(d_in, d_out, context_lenght=context_lenght, dropout=dropout, qkv_bias=qkv_bias)
                for _ in range(num_heads)
            ]
        )
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


In [ ]:
torch.manual_seed(123)
context_lenght = batch.shape[1]
d_in, d_out = 3, 2